# Model Save/Load and Graph Input Format Tutorial

This tutorial covers two topics:

**Part 1: Model Save/Load**
- `torch.save` / `torch.load` for `state_dict` (recommended)
- Full model save
- `ModelCheckpointCallback`: Automatic checkpointing during training
- Loading pretrained weights and fine-tuning (transfer learning)

**Part 2: Graph Input Formats** (porting Keras concepts to PyTorch/PyG)
- Padded tensors via `MemoryGraphList.tensor()`
- PyG disjoint format via `MemoryGraphList.to_pyg_list()` + DataLoader
- Comparison and equivalence demonstration

All `kgcnn_torch` models are standard `nn.Module` subclasses, so standard PyTorch serialization works directly.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
import tempfile

In [ ]:
from kgcnn_torch.models.gcn import GCNModel

# Create a model for demonstration
model_config = {
    "node_dim": 64,
    "depth": 3,
    "gcn_units": 100,
    "gcn_activation": "relu",
    "node_pooling": "sum",
    "output_units": [64, 32],
    "output_activation": "relu",
    "output_final_activation": "linear",
    "num_targets": 1,
    "output_embedding": "graph",
    "use_node_embedding": True,
    "num_embeddings": 95,
}

model = GCNModel(**model_config)
print(model)

## Part 1: Model Save/Load

### 1. Saving and Loading state_dict (Recommended)

The recommended approach is to save only the model's `state_dict()` -- a dictionary mapping parameter names to tensors. This is portable, flexible, and avoids serialization issues.

In [ ]:
# Inspect the state_dict
state_dict = model.state_dict()
print("Number of parameter groups:", len(state_dict))
print("\nParameter names and shapes:")
for name, param in state_dict.items():
    print(f"  {name}: {param.shape}")

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "model_weights.pt")
    
    # Save
    torch.save(model.state_dict(), path)
    print(f"Saved state_dict to {path}")
    print(f"File size: {os.path.getsize(path) / 1024:.1f} KB")
    
    # Load into a new model with the SAME architecture
    model_loaded = GCNModel(**model_config)
    model_loaded.load_state_dict(torch.load(path, weights_only=True))
    model_loaded.eval()
    
    # Verify weights match
    for (n1, p1), (n2, p2) in zip(
        model.state_dict().items(), model_loaded.state_dict().items()
    ):
        assert torch.equal(p1, p2), f"Mismatch at {n1}"
    print("All parameters match after loading.")

#### 1.1 Saving/Loading Optimizer State

To resume training, save both the model and optimizer state_dict.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, "checkpoint.pt")
    
    # Save both model and optimizer
    torch.save({
        "epoch": 50,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": 0.05,
    }, checkpoint_path)
    print(f"Saved checkpoint to {checkpoint_path}")
    
    # Restore
    model_resume = GCNModel(**model_config)
    optimizer_resume = torch.optim.Adam(model_resume.parameters(), lr=1e-3)
    
    checkpoint = torch.load(checkpoint_path, weights_only=False)
    model_resume.load_state_dict(checkpoint["model_state_dict"])
    optimizer_resume.load_state_dict(checkpoint["optimizer_state_dict"])
    
    print(f"Resumed from epoch {checkpoint['epoch']}, loss={checkpoint['train_loss']}")
    print(f"Optimizer LR: {optimizer_resume.param_groups[0]['lr']}")

### 2. Saving the Full Model

You can also save the entire model object using `torch.save(model, path)`. This serializes the model architecture along with the weights using Python's pickle module.

**Note**: This approach is less portable because it depends on the exact class definition being importable at load time.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "full_model.pt")
    
    # Save full model
    torch.save(model, path)
    print(f"Saved full model to {path}")
    print(f"File size: {os.path.getsize(path) / 1024:.1f} KB")
    
    # Load full model -- no need to recreate architecture
    model_full = torch.load(path, weights_only=False)
    model_full.eval()
    
    print(f"Loaded model type: {type(model_full).__name__}")
    print(f"Number of parameters: {sum(p.numel() for p in model_full.parameters())}")

#### Comparison of Save Methods

| Method | Pros | Cons |
|---|---|---|
| `state_dict` only | Portable, flexible, small file size | Must know architecture to reload |
| Full model | No need to know architecture | Less portable, depends on class imports |
| Checkpoint (model + optimizer + metadata) | Full training state for resumption | Larger files |

### 3. ModelCheckpointCallback

The `ModelCheckpointCallback` automatically saves model weights during training, either at every epoch or only when a monitored metric improves. It integrates with the `fit()` training loop.

In [ ]:
from kgcnn_torch.training.callbacks import ModelCheckpointCallback, EarlyStoppingCallback
from kgcnn_torch.training.trainer import fit
from torch_geometric.loader import DataLoader


In [ ]:
# Use real FreeSolv data for checkpoint and early-stopping demos
from sklearn.model_selection import train_test_split
from kgcnn_torch.data.datasets.FreeSolvDataset import FreeSolvDataset

freesolv = FreeSolvDataset()
print(f"FreeSolv dataset: {len(freesolv)} molecules")

indices = np.arange(len(freesolv))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_data = freesolv[torch.tensor(train_idx).long()]
val_data = freesolv[torch.tensor(val_idx).long()]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, "best_model.pt")
    
    # Create model
    model = GCNModel(
        node_dim=32, depth=2, gcn_units=32,
        output_units=[16], output_final_activation="linear",
        num_targets=1
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # ModelCheckpointCallback: save best model based on val_loss
    checkpoint_cb = ModelCheckpointCallback(
        filepath=checkpoint_path,
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    )
    
    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        loss_fn=nn.MSELoss(),
        epochs=30,
        verbose=1,
        callbacks=[checkpoint_cb],
    )
    
    # Load the best checkpoint
    print(f"\nCheckpoint exists: {os.path.exists(checkpoint_path)}")
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    print("Loaded best model weights from checkpoint.")

In [ ]:
# ModelCheckpointCallback with epoch formatting in filepath
with tempfile.TemporaryDirectory() as tmpdir:
    model = GCNModel(
        node_dim=32, depth=2, gcn_units=32,
        output_units=[16], output_final_activation="linear",
        num_targets=1
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    # Save every epoch with formatted name
    checkpoint_cb = ModelCheckpointCallback(
        filepath=os.path.join(tmpdir, "model_epoch_{epoch}.pt"),
        monitor="val_loss",
        save_best_only=False,  # save every epoch
        save_weights_only=True,
        verbose=0,
    )
    
    history = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        loss_fn=nn.MSELoss(),
        epochs=5,
        verbose=0,
        callbacks=[checkpoint_cb],
    )
    
    # List saved checkpoints
    saved_files = sorted([f for f in os.listdir(tmpdir) if f.endswith('.pt')])
    print("Saved checkpoint files:")
    for f in saved_files:
        size = os.path.getsize(os.path.join(tmpdir, f)) / 1024
        print(f"  {f} ({size:.1f} KB)")

### 4. Loading Pretrained Weights and Fine-Tuning

Transfer learning with `kgcnn_torch` models follows standard PyTorch patterns:
1. Load a pretrained model's `state_dict`
2. Optionally freeze some layers
3. Replace or modify the output head
4. Fine-tune on a new task

In [ ]:
# Step 1: "Pretrain" a model (simulate with quick training)
pretrained_config = {
    "node_dim": 32, "depth": 2, "gcn_units": 64,
    "output_units": [32], "output_final_activation": "linear",
    "num_targets": 1
}

pretrained_model = GCNModel(**pretrained_config)
optimizer = torch.optim.Adam(pretrained_model.parameters(), lr=1e-3)

# Quick training
history = fit(
    model=pretrained_model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=nn.MSELoss(),
    epochs=20,
    verbose=0,
)
print(f"Pretrained model final loss: {history['train_loss'][-1]:.4f}")

In [ ]:
# Step 2: Save the pretrained weights
with tempfile.TemporaryDirectory() as tmpdir:
    pretrained_path = os.path.join(tmpdir, "pretrained.pt")
    torch.save(pretrained_model.state_dict(), pretrained_path)
    
    # Step 3: Create a new model with the same backbone but different output
    finetune_config = pretrained_config.copy()
    finetune_config["num_targets"] = 3  # Different number of targets
    finetune_model = GCNModel(**finetune_config)
    
    # Step 4: Load pretrained weights, filtering out size-mismatched layers.
    # Note: strict=False only skips missing/unexpected keys, NOT shape mismatches.
    # We must manually filter keys with incompatible shapes.
    pretrained_state = torch.load(pretrained_path, weights_only=True)
    model_state = finetune_model.state_dict()
    
    compatible_state = {}
    size_mismatched = []
    for k, v in pretrained_state.items():
        if k in model_state and v.shape == model_state[k].shape:
            compatible_state[k] = v
        elif k in model_state:
            size_mismatched.append(k)
    
    missing, unexpected = finetune_model.load_state_dict(compatible_state, strict=False)
    
    print("Size-mismatched keys (skipped):")
    for k in size_mismatched:
        print(f"  {k}: pretrained {pretrained_state[k].shape} vs new {model_state[k].shape}")
    print(f"\nMissing keys (new layers not in pretrained):")
    for k in missing:
        if k not in size_mismatched:
            print(f"  {k}")
    print(f"\nUnexpected keys (pretrained layers not in new model):")
    for k in unexpected:
        print(f"  {k}")

In [ ]:
# Step 5: Freeze backbone, train only the output head
# Freeze all parameters first
for param in finetune_model.parameters():
    param.requires_grad = False

# Unfreeze the output MLP
for param in finetune_model.output_mlp.parameters():
    param.requires_grad = True

# Count trainable vs frozen parameters
trainable = sum(p.numel() for p in finetune_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in finetune_model.parameters())
print(f"Trainable parameters: {trainable} / {total} ({100*trainable/total:.1f}%)")

In [ ]:
# Step 6: Fine-tune with a lower learning rate
# Only pass trainable parameters to the optimizer
finetune_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, finetune_model.parameters()),
    lr=1e-4  # Lower LR for fine-tuning
)

print("Fine-tuning optimizer created with lr=1e-4")
print(f"Number of parameter groups: {len(finetune_optimizer.param_groups)}")

#### 4.1 Partial Weight Loading

Sometimes you want to load only certain layers from a pretrained model. You can filter the `state_dict` manually.

In [ ]:
# Load only GCN convolutional layers (skip embedding and output)
new_model = GCNModel(**pretrained_config)

pretrained_sd = pretrained_model.state_dict()
new_sd = new_model.state_dict()

# Filter: only load parameters from 'convs' and 'dense_in'
partial_sd = {
    k: v for k, v in pretrained_sd.items()
    if k.startswith("convs.") or k.startswith("dense_in.")
}

# Update new model's state_dict with the partial one
new_sd.update(partial_sd)
new_model.load_state_dict(new_sd)

print(f"Loaded {len(partial_sd)} parameter groups from pretrained model:")
for k in partial_sd:
    print(f"  {k}")

#### 4.2 Unfreezing Gradually

A common fine-tuning strategy is to first train only the head, then gradually unfreeze deeper layers.

In [ ]:
finetune_model2 = GCNModel(**pretrained_config)

# Phase 1: Freeze everything except output_mlp
for param in finetune_model2.parameters():
    param.requires_grad = False
for param in finetune_model2.output_mlp.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in finetune_model2.parameters() if p.requires_grad)
total = sum(p.numel() for p in finetune_model2.parameters())
print(f"Phase 1 - Trainable: {trainable}/{total}")

# Phase 2: Also unfreeze the last GCN conv layer
for param in finetune_model2.convs[-1].parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in finetune_model2.parameters() if p.requires_grad)
print(f"Phase 2 - Trainable: {trainable}/{total}")

# Phase 3: Unfreeze all layers
for param in finetune_model2.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in finetune_model2.parameters() if p.requires_grad)
print(f"Phase 3 - Trainable: {trainable}/{total}")

### 5. Early Stopping with Best Model Restore

The `EarlyStoppingCallback` can automatically restore the best model weights when training stops.

In [ ]:
from kgcnn_torch.training.callbacks import EarlyStoppingCallback

model = GCNModel(
    node_dim=32, depth=2, gcn_units=32,
    output_units=[16], output_final_activation="linear",
    num_targets=1
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

early_stop = EarlyStoppingCallback(
    patience=8,
    monitor="val_loss",
    mode="min",
    restore_best_weights=True,  # restore model to best epoch
    verbose=1,
)

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    loss_fn=nn.MSELoss(),
    epochs=100,
    verbose=1,
    callbacks=[early_stop],
)

print(f"\nTraining stopped after {len(history['train_loss'])} epochs")
print(f"Best val_loss was at epoch {early_stop._best_epoch + 1}")

---
## Part 2: Graph Input Formats

In graph neural networks, batching variable-size graphs is handled differently than standard tensor batching.
There are three main representations:

1. **Padded tensors**: Each graph is padded to the same size, stacked into a batch tensor. Requires masks to ignore padding. Used by `MemoryGraphList.tensor()`.

2. **Variable-length (ragged)**: Each graph has different sizes stored as a list. In TensorFlow this uses `RaggedTensor`; in PyTorch there is no direct equivalent.

3. **Disjoint (PyG native)**: All graphs in a batch are concatenated into one large graph with a `batch` vector indicating which nodes belong to which graph. This is PyG's native format and the most efficient for PyTorch.

In `kgcnn-torch`, the recommended approach is the **disjoint format** via `to_pyg_list()` + PyG DataLoader.

### Load a Real Dataset

We use the FreeSolv dataset (642 molecules) to demonstrate both formats.

In [ ]:
if "freesolv" not in globals():
    from kgcnn_torch.data.datasets.FreeSolvDataset import FreeSolvDataset
    freesolv = FreeSolvDataset()

print(f"FreeSolv dataset: {len(freesolv)} molecules")
print(f"Example graph: {freesolv[0]}")


### Format 1: Padded Tensors

The `MemoryGraphList.tensor()` method pads variable-length graph properties to the maximum
size in the dataset and stacks them into batch tensors. This is analogous to the Keras "padded" format.

We demonstrate this using the KGCNN internal `MemoryGraphList`, which is the raw data before
conversion to PyG format.

In [ ]:
from kgcnn_torch.data.base import MemoryGraphList
from kgcnn_torch.graph.base import GraphDict

# Convert a few real FreeSolv samples into MemoryGraphList for padded tensor demo
demo_graphs = MemoryGraphList()
for i in range(5):
    sample = freesolv[i]
    node_number = sample.z.detach().cpu().numpy()
    edge_indices = sample.edge_index.detach().cpu().numpy().T[:, [1, 0]]
    graph_label = sample.y.detach().cpu().numpy().reshape(-1)[:1].astype(np.float32)

    demo_graphs.append(GraphDict({
        "node_number": node_number,
        "edge_indices": edge_indices,
        "graph_labels": graph_label,
    }))

# Define input specification (like Keras model config inputs)
inputs_spec = [
    {"shape": (None,), "name": "node_number", "dtype": "int64"},
    {"shape": (None, 2), "name": "edge_indices", "dtype": "int64"},
]
output_spec = {"shape": (1,), "name": "graph_labels", "dtype": "float32"}

# Padded tensor conversion
x_padded = demo_graphs.tensor(inputs_spec)
y_padded = demo_graphs.tensor(output_spec)

print("Padded format:")
for i, (inp, spec) in enumerate(zip(x_padded, inputs_spec)):
    print(f"  {spec['name']}: shape={inp.shape}, dtype={inp.dtype}")
print(f"  labels: shape={y_padded.shape}")
print(f"\nNote: Graphs with fewer nodes/edges are zero-padded to the max size.")
print(f"Graph sizes: {[len(g['node_number']) for g in demo_graphs]}")
print(f"Padded node_number batch shape: {x_padded[0].shape} (padded to max={x_padded[0].shape[1]})")


### Format 2: Disjoint (PyG Native)

The `MemoryGraphList.to_pyg_list()` method converts each graph to a PyG `Data` object.
When batched by PyG's `DataLoader`, all graphs are concatenated into a single disjoint graph:

- Node features from all graphs are concatenated: `[x_0, x_1, ..., x_B]`
- Edge indices are offset and concatenated
- A `batch` vector indicates which graph each node belongs to

This is the most memory-efficient format and is natively supported by all PyG operations.

In [ ]:
from torch_geometric.loader import DataLoader

# Convert to PyG list
demo_graphs.map_list(method="set_edge_weights_uniform")
demo_graphs.map_list(method="normalize_edge_weights_sym")
pyg_demo = demo_graphs.to_pyg_list()

print("Individual PyG Data objects:")
for i, d in enumerate(pyg_demo):
    print(f"  Graph {i}: {d}")

# Batch with PyG DataLoader
loader = DataLoader(pyg_demo, batch_size=3)
batch = next(iter(loader))

print(f"\nBatched (disjoint) graph:")
print(f"  batch.z shape: {batch.z.shape}  (all nodes concatenated)")
print(f"  batch.edge_index shape: {batch.edge_index.shape}  (all edges concatenated, indices offset)")
print(f"  batch.batch: {batch.batch}  (graph assignment for each node)")
print(f"  batch.y shape: {batch.y.shape}  (labels stacked)")
print(f"\nNo padding needed -- variable-size graphs are naturally handled!")

### Training with the Full FreeSolv Dataset

The PyG disjoint format is the standard way to train GNN models in `kgcnn-torch`.
The `FreeSolvDataset` class returns PyG `Data` objects directly,
ready for use with `DataLoader` and `fit()`.


In [ ]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Split dataset
indices = np.arange(len(freesolv))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_data = freesolv[torch.tensor(train_idx).long()]
val_data = freesolv[torch.tensor(val_idx).long()]

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

# Inspect a batch
batch = next(iter(train_loader))
print(f"Train batch: {batch}")
print(f"  Nodes: {batch.z.shape[0]}, Edges: {batch.edge_index.shape[1]}")
print(f"  Graphs in batch: {batch.num_graphs}")

In [ ]:
# Train a GCN model on FreeSolv using disjoint format
model_disjoint = GCNModel(
    node_dim=64, depth=3, gcn_units=64,
    output_units=[32], output_activation="relu",
    output_final_activation="linear",
    num_targets=1, output_embedding="graph",
    use_node_embedding=True, num_embeddings=95,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

history = fit(
    model=model_disjoint,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=torch.optim.Adam(model_disjoint.parameters(), lr=1e-3),
    loss_fn=nn.L1Loss(),
    epochs=100,
    device=device,
    verbose=1,
)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(history["train_loss"], label="Train MAE")
ax.plot(history["val_loss"], label="Val MAE")
ax.set_xlabel("Epoch")
ax.set_ylabel("MAE Loss")
ax.set_title("GCN on FreeSolv (PyG Disjoint Format)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final train MAE: {history['train_loss'][-1]:.4f}")
print(f"Final val MAE: {history['val_loss'][-1]:.4f}")

### Format Comparison Summary

| Format | Method | Pros | Cons | Use in kgcnn-torch |
|---|---|---|---|---|
| **Padded** | `MemoryGraphList.tensor()` | Simple, compatible with standard NN | Memory waste from padding, needs masks | For inspection/debugging |
| **Disjoint (PyG)** | `to_pyg_list()` + `DataLoader` | No padding waste, native PyG support, efficient | Requires scatter/segment ops | **Recommended for training** |
| **Ragged** | TF RaggedTensor | No padding, variable-length | TensorFlow-only, not available in PyTorch | N/A in PyTorch |

In `kgcnn-torch`, always use the **disjoint (PyG) format** for model training. The padded format
via `tensor()` is available for inspection or compatibility but is not the standard training path.

## Summary

**Part 1: Model Save/Load**

| Method | Use Case | Code |
|---|---|---|
| Save `state_dict` | Portable weight storage | `torch.save(model.state_dict(), path)` |
| Load `state_dict` | Restore weights to same architecture | `model.load_state_dict(torch.load(path))` |
| Save full model | Quick save (less portable) | `torch.save(model, path)` |
| Checkpoint callback | Auto-save during training | `ModelCheckpointCallback(filepath=...)` |
| Partial loading | Transfer learning | `model.load_state_dict(state, strict=False)` |
| Freeze layers | Fine-tuning | `param.requires_grad = False` |
| Early stopping + restore | Best model selection | `EarlyStoppingCallback(restore_best_weights=True)` |

**Part 2: Graph Input Formats**

- Use `MemoryGraphList.tensor()` for padded batch tensors (inspection/compatibility)
- Use `MemoryGraphList.to_pyg_list()` + PyG `DataLoader` for disjoint format (recommended for training)
- PyG DataLoader automatically handles variable-size graphs via concatenation + batch vector